In [1]:
!apt-get update -qq
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!pip uninstall -y pyspark
!pip install pyspark==3.4.1 findspark

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Found existing installation: pyspark 4.0.2
Uninstalling pyspark-4.0.2:
  Successfully uninstalled pyspark-4.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.8/310.8 MB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 13.2 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.4.1-py2.py3-none-any.whl size=311285391 sha256=45e55742a1d5a6e9c779c6f8f3b7795701fe4185617e024feeafc612fb63f2e0
  Stored in directory: /root/.cache/pip/wheels/8d/95/1d/739a17bda5d6a1c3c6f60eed9a82f600ab0d9fcd4c601ce0da
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9
ERROR: pip's dependency resolver does not current

In [5]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["PYSPARK_PYTHON"] = "python3"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python3"

In [4]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("BankingAnalytics").master("local[*]").getOrCreate()
print("Spark Started Successfully")
print("Version:", spark.version)

Spark Started Successfully
Version: 3.4.1


In [6]:
customers_csv = """customer_id,name,city,age,account_type,signup_date
1,Rahul,Hyderabad,29,Savings,2023-01-10
2,Sneha,Bangalore,32,Current,2023-02-12
3,Arjun,Mumbai,27,Savings,2023-03-14
4,Priya,Delhi,35,Savings,2023-04-15
5,Karan,Chennai,30,Current,2023-05-11
6,Meera,Hyderabad,31,Savings,2023-06-10
7,Amit,Pune,38,Current,2023-06-22
8,Neha,Delhi,26,Savings,2023-07-10
9,Divya,Bangalore,28,Savings,2023-07-15
10,Vikram,Mumbai,42,Current,2023-08-01
11,Farhan,Hyderabad,34,Savings,2023-08-10
12,Simran,Delhi,25,Savings,2023-08-21
"""

accounts_csv = """account_id,customer_id,branch,balance
1001,1,Hyderabad Main,85000
1002,2,Bangalore Central,120000
1003,3,Mumbai West,45000
1004,4,Delhi North,95000
1005,5,Chennai South,60000
1006,6,Hyderabad Main,150000
1007,7,Pune East,30000
1008,8,Delhi North,70000
1009,9,Bangalore Central,110000
1010,10,Mumbai West,200000
1011,11,Hyderabad Main,50000
1012,12,Delhi North,40000
"""

transactions_csv = """txn_id,account_id,txn_type,amount,txn_date
1,1001,Credit,25000,2024-03-01
2,1002,Debit,15000,2024-03-01
3,1003,Credit,10000,2024-03-02
4,1004,Debit,5000,2024-03-02
5,1005,Credit,30000,2024-03-03
6,1006,Debit,20000,2024-03-03
7,1007,Credit,8000,2024-03-04
8,1008,Debit,12000,2024-03-04
9,1009,Credit,40000,2024-03-05
10,1010,Debit,35000,2024-03-05
11,1001,Debit,7000,2024-03-06
12,1002,Credit,18000,2024-03-06
13,1006,Credit,50000,2024-03-07
14,1010,Credit,60000,2024-03-07
15,1011,Debit,9000,2024-03-08
16,1012,Credit,16000,2024-03-08
17,1003,Debit,4000,2024-03-09
18,1004,Credit,22000,2024-03-09
19,1005,Debit,11000,2024-03-10
20,1009,Debit,14000,2024-03-10
"""

branches_csv = """branch,region,manager
Hyderabad Main,South,Ramesh
Bangalore Central,South,Leena
Mumbai West,West,Joseph
Delhi North,North,Sara
Chennai South,South,Kumar
Pune East,West,Anita
"""

logs_txt = """Rahul login
Sneha login
Rahul transfer
Arjun login
Priya withdrawal
Rahul logout
Meera login
Vikram transfer
Divya login
Farhan login
Simran withdrawal
Neha login
Amit deposit
Karan login
Meera transfer
Vikram login
Rahul deposit
Sneha withdrawal
Farhan transfer
Divya logout
"""

customer_profiles_json = """[
  {
    "customer_id": 1,
    "name": "Rahul",
    "contact": {"email": "rahul@mail.com", "phone": "9000011111"},
    "services": ["UPI", "Credit Card", "Net Banking"]
  },
  {
    "customer_id": 2,
    "name": "Sneha",
    "contact": {"email": "sneha@mail.com", "phone": "9000022222"},
    "services": ["UPI", "Debit Card"]
  },
  {
    "customer_id": 3,
    "name": "Arjun",
    "contact": {"email": "arjun@mail.com", "phone": "9000033333"},
    "services": ["Net Banking", "Loan"]
  },
  {
    "customer_id": 6,
    "name": "Meera",
    "contact": {"email": "meera@mail.com", "phone": "9000066666"},
    "services": ["UPI", "Credit Card", "Loan"]
  },
  {
    "customer_id": 10,
    "name": "Vikram",
    "contact": {"email": "vikram@mail.com", "phone": "9000101010"},
    "services": ["Net Banking", "Wealth"]
  }
]"""

with open("bank_customers.csv", "w") as f:
    f.write(customers_csv)

with open("accounts.csv", "w") as f:
    f.write(accounts_csv)

with open("transactions.csv", "w") as f:
    f.write(transactions_csv)

with open("branches.csv", "w") as f:
    f.write(branches_csv)

with open("bank_logs.txt", "w") as f:
    f.write(logs_txt)

with open("customer_profiles.json", "w") as f:
    f.write(customer_profiles_json)

print("Banking datasets created successfully")

Banking datasets created successfully


In [7]:
customers = spark.read.csv("bank_customers.csv", header=True, inferSchema=True)
accounts = spark.read.csv("accounts.csv", header=True, inferSchema=True)
transactions = spark.read.csv("transactions.csv", header=True, inferSchema=True)
branches = spark.read.csv("branches.csv", header=True, inferSchema=True)
profiles = spark.read.json("customer_profiles.json", multiLine=True)

In [8]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [10]:
#1
customers.show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
+-----------+------+---------+---+------------+-----------+



In [12]:
#2
accounts.show()

+----------+-----------+-----------------+-------+
|account_id|customer_id|           branch|balance|
+----------+-----------+-----------------+-------+
|      1001|          1|   Hyderabad Main|  85000|
|      1002|          2|Bangalore Central| 120000|
|      1003|          3|      Mumbai West|  45000|
|      1004|          4|      Delhi North|  95000|
|      1005|          5|    Chennai South|  60000|
|      1006|          6|   Hyderabad Main| 150000|
|      1007|          7|        Pune East|  30000|
|      1008|          8|      Delhi North|  70000|
|      1009|          9|Bangalore Central| 110000|
|      1010|         10|      Mumbai West| 200000|
|      1011|         11|   Hyderabad Main|  50000|
|      1012|         12|      Delhi North|  40000|
+----------+-----------+-----------------+-------+



In [13]:
#3
transactions.show()

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|     1|      1001|  Credit| 25000|2024-03-01|
|     2|      1002|   Debit| 15000|2024-03-01|
|     3|      1003|  Credit| 10000|2024-03-02|
|     4|      1004|   Debit|  5000|2024-03-02|
|     5|      1005|  Credit| 30000|2024-03-03|
|     6|      1006|   Debit| 20000|2024-03-03|
|     7|      1007|  Credit|  8000|2024-03-04|
|     8|      1008|   Debit| 12000|2024-03-04|
|     9|      1009|  Credit| 40000|2024-03-05|
|    10|      1010|   Debit| 35000|2024-03-05|
|    11|      1001|   Debit|  7000|2024-03-06|
|    12|      1002|  Credit| 18000|2024-03-06|
|    13|      1006|  Credit| 50000|2024-03-07|
|    14|      1010|  Credit| 60000|2024-03-07|
|    15|      1011|   Debit|  9000|2024-03-08|
|    16|      1012|  Credit| 16000|2024-03-08|
|    17|      1003|   Debit|  4000|2024-03-09|
|    18|      1004|  Credit| 22000|2024-03-09|
|    19|     

In [14]:
#4
branches.show()

+-----------------+------+-------+
|           branch|region|manager|
+-----------------+------+-------+
|   Hyderabad Main| South| Ramesh|
|Bangalore Central| South|  Leena|
|      Mumbai West|  West| Joseph|
|      Delhi North| North|   Sara|
|    Chennai South| South|  Kumar|
|        Pune East|  West|  Anita|
+-----------------+------+-------+



In [15]:
#5
customers.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- account_type: string (nullable = true)
 |-- signup_date: date (nullable = true)



In [16]:
#6
transactions.printSchema()

root
 |-- txn_id: integer (nullable = true)
 |-- account_id: integer (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- txn_date: date (nullable = true)



In [17]:
#7
customers.count()

12

In [18]:
#8
accounts.count()

12

In [19]:
#9
transactions.count()

20

In [20]:
#10
transactions.show(5)

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|     1|      1001|  Credit| 25000|2024-03-01|
|     2|      1002|   Debit| 15000|2024-03-01|
|     3|      1003|  Credit| 10000|2024-03-02|
|     4|      1004|   Debit|  5000|2024-03-02|
|     5|      1005|  Credit| 30000|2024-03-03|
+------+----------+--------+------+----------+
only showing top 5 rows



In [11]:
#11
customers.select("name","city","account_type").show()

#12
accounts.select("account_id","balance").show()

#13
accounts.withColumnRenamed("balance","current_balance").show()

#14
transactions.withColumnRenamed("txn_type","transaction_type").show()

#15
customers.filter(col("city")=="Hyderabad").show()

#16
customers.filter(col("age")>30).show()

#17
customers.filter(col("account_type")=="Savings").show()

#18
accounts.filter(col("balance")>100000).show()

#19
transactions.filter(col("txn_type")=="Credit").show()

#20
transactions.filter(col("amount")>20000).show()

+------+---------+------------+
|  name|     city|account_type|
+------+---------+------------+
| Rahul|Hyderabad|     Savings|
| Sneha|Bangalore|     Current|
| Arjun|   Mumbai|     Savings|
| Priya|    Delhi|     Savings|
| Karan|  Chennai|     Current|
| Meera|Hyderabad|     Savings|
|  Amit|     Pune|     Current|
|  Neha|    Delhi|     Savings|
| Divya|Bangalore|     Savings|
|Vikram|   Mumbai|     Current|
|Farhan|Hyderabad|     Savings|
|Simran|    Delhi|     Savings|
+------+---------+------------+

+----------+-------+
|account_id|balance|
+----------+-------+
|      1001|  85000|
|      1002| 120000|
|      1003|  45000|
|      1004|  95000|
|      1005|  60000|
|      1006| 150000|
|      1007|  30000|
|      1008|  70000|
|      1009| 110000|
|      1010| 200000|
|      1011|  50000|
|      1012|  40000|
+----------+-------+

+----------+-----------+-----------------+---------------+
|account_id|customer_id|           branch|current_balance|
+----------+-----------+--------

In [21]:
#21
customers.orderBy("age").show()

#22
customers.orderBy(col("age").desc()).show()

#23
accounts.orderBy(col("balance").desc()).show()

#24
accounts.orderBy(col("balance").desc()).show(5)

#25
accounts.orderBy("balance").show(5)

#26
transactions.orderBy(col("amount").desc()).show()

#27
transactions.orderBy(col("amount").desc()).show(3)

#28
transactions.orderBy("txn_date").show()

#29
customers.orderBy("city","age").show()

#30
transactions.orderBy(col("txn_date").desc()).show(5)

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
+-----------+------+---------+---+------------+-----------+

+-----------+------+---------+---+-----

In [22]:
#31
accounts.select(sum("balance")).show()

#32
accounts.select(avg("balance")).show()

#33
accounts.select(max("balance")).show()

#34
accounts.select(min("balance")).show()

#35
customers.groupBy("city").count().show()

#36
customers.groupBy("account_type").count().show()

#37
transactions.groupBy("txn_type").count().show()

#38
transactions.filter(col("txn_type")=="Credit").select(sum("amount")).show()

#39
transactions.filter(col("txn_type")=="Debit").select(sum("amount")).show()

#40
transactions.select(avg("amount")).show()

+------------+
|sum(balance)|
+------------+
|     1055000|
+------------+

+-----------------+
|     avg(balance)|
+-----------------+
|87916.66666666667|
+-----------------+

+------------+
|max(balance)|
+------------+
|      200000|
+------------+

+------------+
|min(balance)|
+------------+
|       30000|
+------------+

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    1|
|   Mumbai|    2|
|     Pune|    1|
|    Delhi|    3|
|Hyderabad|    3|
+---------+-----+

+------------+-----+
|account_type|count|
+------------+-----+
|     Savings|    8|
|     Current|    4|
+------------+-----+

+--------+-----+
|txn_type|count|
+--------+-----+
|  Credit|   10|
|   Debit|   10|
+--------+-----+

+-----------+
|sum(amount)|
+-----------+
|     279000|
+-----------+

+-----------+
|sum(amount)|
+-----------+
|     132000|
+-----------+

+-----------+
|avg(amount)|
+-----------+
|    20550.0|
+-----------+



In [23]:
#41
accounts.groupBy("branch").sum("balance").show()

#42
accounts.groupBy("branch").avg("balance").show()

#43
accounts.groupBy("branch").count().show()

#44
transactions.groupBy("account_id").sum("amount").show()

#45
transactions.groupBy("account_id").count().show()

#46
transactions.groupBy("txn_type").sum("amount").show()

#47
transactions.groupBy("txn_date").count().show()

#48
customers.groupBy("city").avg("age").show()

#49
customers.join(accounts,"customer_id").groupBy("account_type").sum("balance").show()

#50
customers.groupBy("city","account_type").count().show()

+-----------------+------------+
|           branch|sum(balance)|
+-----------------+------------+
|      Delhi North|      205000|
|   Hyderabad Main|      285000|
|        Pune East|       30000|
|      Mumbai West|      245000|
|    Chennai South|       60000|
|Bangalore Central|      230000|
+-----------------+------------+

+-----------------+-----------------+
|           branch|     avg(balance)|
+-----------------+-----------------+
|      Delhi North|68333.33333333333|
|   Hyderabad Main|          95000.0|
|        Pune East|          30000.0|
|      Mumbai West|         122500.0|
|    Chennai South|          60000.0|
|Bangalore Central|         115000.0|
+-----------------+-----------------+

+-----------------+-----+
|           branch|count|
+-----------------+-----+
|      Delhi North|    3|
|   Hyderabad Main|    3|
|        Pune East|    1|
|      Mumbai West|    2|
|    Chennai South|    1|
|Bangalore Central|    2|
+-----------------+-----+

+----------+-----------+
|a

In [24]:
#51
customers.join(accounts,"customer_id").show()

#52
customers.join(accounts,"customer_id").select("name","city","branch","balance").show()

#53
accounts.join(transactions,"account_id").show()

#54
accounts.join(transactions,"account_id").select("account_id","txn_type","amount","balance").show()

#55
accounts.join(branches,"branch").show()

#56
accounts.join(branches,"branch").select("branch","region","manager","balance").show()

#57
customers.join(accounts,"customer_id").join(transactions,"account_id").show()

#58
customers.join(accounts,"customer_id").join(transactions,"account_id").select("name","city","txn_type","amount","txn_date").show()

#59
customers.join(accounts,"customer_id").join(branches,"branch").join(transactions,"account_id").show()

#60
customers.join(accounts,"customer_id").join(transactions,"account_id").groupBy("name").sum("amount").show()

+-----------+------+---------+---+------------+-----------+----------+-----------------+-------+
|customer_id|  name|     city|age|account_type|signup_date|account_id|           branch|balance|
+-----------+------+---------+---+------------+-----------+----------+-----------------+-------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|      1001|   Hyderabad Main|  85000|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|      1002|Bangalore Central| 120000|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|      1003|      Mumbai West|  45000|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|      1004|      Delhi North|  95000|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|      1005|    Chennai South|  60000|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|      1006|   Hyderabad Main| 150000|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|      1007|        Pune East|  30000|
|          8|  Neha|    Delhi|

In [25]:
#61
accounts.withColumn("balance_in_lakhs", col("balance")/100000).show()

#62
accounts.withColumn("bank_name", lit("BotCampus Bank")).show()

#63
accounts.withColumn("annual_service_fee", col("balance")*0.01).show()

#64
accounts.withColumn("annual_service_fee", col("balance")*0.01).withColumn("net_balance", col("balance")-col("annual_service_fee")).show()

#65
accounts.withColumn("is_high_balance", col("balance")>100000).show()

#66
transactions.withColumn("txn_amount_in_k", col("amount")/1000).show()

#67
customers.withColumn("country", lit("India")).show()

#68
customers.withColumn("customer_label", concat_ws(" - ","name","city")).show()

#69
accounts.join(branches,"branch").withColumn("branch_label", concat_ws(" - ","branch","region")).show()

#70
transactions.withColumn("risk_flag", when(col("amount")>40000,"High").otherwise("Normal")).show()

+----------+-----------+-----------------+-------+----------------+
|account_id|customer_id|           branch|balance|balance_in_lakhs|
+----------+-----------+-----------------+-------+----------------+
|      1001|          1|   Hyderabad Main|  85000|            0.85|
|      1002|          2|Bangalore Central| 120000|             1.2|
|      1003|          3|      Mumbai West|  45000|            0.45|
|      1004|          4|      Delhi North|  95000|            0.95|
|      1005|          5|    Chennai South|  60000|             0.6|
|      1006|          6|   Hyderabad Main| 150000|             1.5|
|      1007|          7|        Pune East|  30000|             0.3|
|      1008|          8|      Delhi North|  70000|             0.7|
|      1009|          9|Bangalore Central| 110000|             1.1|
|      1010|         10|      Mumbai West| 200000|             2.0|
|      1011|         11|   Hyderabad Main|  50000|             0.5|
|      1012|         12|      Delhi North|  4000

In [26]:
#71
accounts.withColumn("category",when(col("balance")>=100000,"High").when(col("balance")>=50000,"Medium").otherwise("Low")).show()

#72
customers.withColumn("age_group",when(col("age")<30,"Young").when(col("age")<40,"Adult").otherwise("Senior")).show()

#73
transactions.withColumn("txn_category",when(col("amount")>=30000,"Large").when(col("amount")>=10000,"Medium").otherwise("Small")).show()

#74
branches.withColumn("priority",when(col("region")=="South","High Priority").when(col("region")=="North","Medium Priority").otherwise("Watch")).show()

#75
customers.withColumn("label",when(col("account_type")=="Savings","Retail").otherwise("Business")).show()

+----------+-----------+-----------------+-------+--------+
|account_id|customer_id|           branch|balance|category|
+----------+-----------+-----------------+-------+--------+
|      1001|          1|   Hyderabad Main|  85000|  Medium|
|      1002|          2|Bangalore Central| 120000|    High|
|      1003|          3|      Mumbai West|  45000|     Low|
|      1004|          4|      Delhi North|  95000|  Medium|
|      1005|          5|    Chennai South|  60000|  Medium|
|      1006|          6|   Hyderabad Main| 150000|    High|
|      1007|          7|        Pune East|  30000|     Low|
|      1008|          8|      Delhi North|  70000|  Medium|
|      1009|          9|Bangalore Central| 110000|    High|
|      1010|         10|      Mumbai West| 200000|    High|
|      1011|         11|   Hyderabad Main|  50000|  Medium|
|      1012|         12|      Delhi North|  40000|     Low|
+----------+-----------+-----------------+-------+--------+

+-----------+------+---------+---+-----

In [36]:
#76
customers = customers.withColumn("signup_date", to_date("signup_date"))

#77
customers.withColumn("signup_year", year("signup_date")).show()

#78
customers.withColumn("signup_month", month("signup_date")).show()

#79
transactions = transactions.withColumn("txn_date", to_date("txn_date"))

#80
transactions.withColumn("txn_month", month("txn_date")).show()

#81
transactions.groupBy("txn_date").count().show()

#82
transactions.groupBy(month("txn_date")).count().show()

#83
customers.filter(col("signup_date")>"2023-06-01").show()

#84
customers.withColumn("days_since_signup", datediff(current_date(),"signup_date")).show()

#85
transactions.withColumn("days_since_txn", datediff(current_date(),"txn_date")).show()

+-----------+------+---------+---+------------+-----------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|signup_year|
+-----------+------+---------+---+------------+-----------+-----------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|       2023|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|       2023|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|       2023|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|       2023|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|       2023|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|       2023|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|       2023|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|       2023|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|       2023|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|       2023|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|     

In [37]:
#86
w=Window.partitionBy("city").orderBy(col("balance").desc())
customers.join(accounts,"customer_id").withColumn("rank",rank().over(w)).show()

#87
customers.join(accounts,"customer_id").withColumn("rn",row_number().over(w)).filter("rn=1").show()

#88
w2=Window.partitionBy("txn_type").orderBy(col("amount").desc())
transactions.withColumn("drank",dense_rank().over(w2)).show()

#89
w3=Window.partitionBy("account_id").orderBy(col("amount").desc())
transactions.withColumn("rn",row_number().over(w3)).filter("rn<=2").show()

#90
w4=Window.partitionBy("account_id").orderBy("txn_date")
transactions.withColumn("running_total",sum("amount").over(w4)).show()

+-----------+------+---------+---+------------+-----------+----------+-----------------+-------+----+
|customer_id|  name|     city|age|account_type|signup_date|account_id|           branch|balance|rank|
+-----------+------+---------+---+------------+-----------+----------+-----------------+-------+----+
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|      1002|Bangalore Central| 120000|   1|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|      1009|Bangalore Central| 110000|   2|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|      1005|    Chennai South|  60000|   1|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|      1004|      Delhi North|  95000|   1|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|      1008|      Delhi North|  70000|   2|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|      1012|      Delhi North|  40000|   3|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|      1006|   Hyderabad

In [38]:
#91
accounts.groupBy("branch").sum("balance").withColumn("rank",rank().over(Window.orderBy(col("sum(balance)").desc()))).show()

#92
customers.join(accounts,"customer_id").groupBy("city").sum("balance").withColumn("rank",rank().over(Window.orderBy(col("sum(balance)").desc()))).show()

#93
customers.join(accounts,"customer_id").join(transactions,"account_id").groupBy("name").sum("amount").withColumn("rank",rank().over(Window.orderBy(col("sum(amount)").desc()))).show()

#94
customers.join(accounts,"customer_id").join(transactions,"account_id").groupBy("city","name").sum("amount").withColumn("rank",rank().over(Window.partitionBy("city").orderBy(col("sum(amount)").desc()))).filter("rank=1").show()

#95
customers.join(accounts,"customer_id").join(branches,"branch").withColumn("rank",rank().over(Window.partitionBy("region").orderBy(col("balance").desc()))).filter("rank=1").show()

+-----------------+------------+----+
|           branch|sum(balance)|rank|
+-----------------+------------+----+
|   Hyderabad Main|      285000|   1|
|      Mumbai West|      245000|   2|
|Bangalore Central|      230000|   3|
|      Delhi North|      205000|   4|
|    Chennai South|       60000|   5|
|        Pune East|       30000|   6|
+-----------------+------------+----+

+---------+------------+----+
|     city|sum(balance)|rank|
+---------+------------+----+
|Hyderabad|      285000|   1|
|   Mumbai|      245000|   2|
|Bangalore|      230000|   3|
|    Delhi|      205000|   4|
|  Chennai|       60000|   5|
|     Pune|       30000|   6|
+---------+------------+----+

+------+-----------+----+
|  name|sum(amount)|rank|
+------+-----------+----+
|Vikram|      95000|   1|
| Meera|      70000|   2|
| Divya|      54000|   3|
| Karan|      41000|   4|
| Sneha|      33000|   5|
| Rahul|      32000|   6|
| Priya|      27000|   7|
|Simran|      16000|   8|
| Arjun|      14000|   9|
|  Neh

In [43]:
#96
profiles.show()

#97
profiles.select("customer_id","name",col("contact.email").alias("email"),col("contact.phone").alias("phone")).show()

#98
profiles.select("customer_id","name",explode("services").alias("service")).show()

#99
profiles.select("customer_id","name",explode("services").alias("service")).filter(col("service")=="UPI").show()

#100
profiles.select(explode("services").alias("service")).groupBy("service").count().show()

+--------------------+-----------+------+--------------------+
|             contact|customer_id|  name|            services|
+--------------------+-----------+------+--------------------+
|{rahul@mail.com, ...|          1| Rahul|[UPI, Credit Card...|
|{sneha@mail.com, ...|          2| Sneha|   [UPI, Debit Card]|
|{arjun@mail.com, ...|          3| Arjun| [Net Banking, Loan]|
|{meera@mail.com, ...|          6| Meera|[UPI, Credit Card...|
|{vikram@mail.com,...|         10|Vikram|[Net Banking, Wea...|
+--------------------+-----------+------+--------------------+

+-----------+------+---------------+----------+
|customer_id|  name|          email|     phone|
+-----------+------+---------------+----------+
|          1| Rahul| rahul@mail.com|9000011111|
|          2| Sneha| sneha@mail.com|9000022222|
|          3| Arjun| arjun@mail.com|9000033333|
|          6| Meera| meera@mail.com|9000066666|
|         10|Vikram|vikram@mail.com|9000101010|
+-----------+------+---------------+----------+
